In [ ]:
import os, re
import warnings
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from steering_audit.eval import load_eval_task

%load_ext autoreload
%autoreload 2

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
model_label_mapping = {
    "Llama-3.1-8B-Instruct": "Llama3.1",
    "Ministral-8B-Instruct-2410": "Ministral",
    "Qwen2.5-7B-Instruct": "Qwen2.5", 
    "Saul-7B-Instruct-v1": "Saul-7B",
    "medicine-Llama3-8B": "med-Llama3",
    "finance-Llama3-8B": "fin-Llama3"
}
def load_steering_results(artifact_dir, task_name, concept):
    task = load_eval_task(task_name)
    outputs = []
    for filepath in Path(artifact_dir / f'evaluation/{task_name}').rglob(f'{task_name}*.json'):
        coeff = float(filepath.name.split("_coeff=")[-1].replace(".json", ""))
        if task_name == "admissions":
            result = task.compute_result_by_group(filepath, group_type=concept)
        else:
            result = task.compute_result_by_group(filepath)
        for group, score in result.items():
            outputs.append({
                "group": group,
                "score": score,
                "coeff": coeff
            })
    return pd.DataFrame.from_records(outputs)


def get_steering_evaluation_results(artifact_dir=None, concept=None, train_dataset=None, task_name="judicial_guilt", model_list=None):
    if artifact_dir is None:
        artifact_dir = f"runs/{concept}-{train_dataset}"
    if model_list is None:
        model_list = os.listdir(artifact_dir)
    
    results = []
    for model in model_list:
        df = load_steering_results(Path(f'{artifact_dir}/{model}'), task_name=task_name, concept=concept)
        if df is None:
            continue
        if task_name.startswith("diversitymedqa"):
            df = df[df.group == "neutral"]

        x = df.coeff.to_numpy()
        y = df["score"].to_numpy()
        slope, intercept, r, p, std_err = stats.linregress(x, y)
        results.append({
            "model": model,
            "task_name": task_name,
            "slope": slope,
            "p": p,
        })
    return pd.DataFrame.from_records(results)


def get_blackbox_evaluation_results(concept, artifact_dir=Path("runs/baseline"), task_name="judicial_guilt", model_list=None):
    if model_list is None:
        model_list = os.listdir(artifact_dir)

    results = []
    task = load_eval_task(task_name)
    for model in model_list:
        for filepath in Path(artifact_dir / f'{model}/evaluation/{task_name}').rglob(f'{task_name}*.json'):
            if re.search(r"_explicit-baseline.json$", filepath.name):
                is_explicit = True
            elif re.search(r"-baseline.json$", filepath.name):
                is_explicit = False

            if task_name == "admissions":
                group_result = task.compute_result_by_group(filepath, group_type=concept)
            else:
                group_result = task.compute_result_by_group(filepath)
            
            if concept == "race":
                group_diff = group_result["black"] - group_result["white"]
            elif concept == "gender":
                group_diff = group_result["female"] - group_result["male"]

            results.append({
                "model": model,
                "task_name": task_name,
                "explicit": is_explicit,
                "group_diff": group_diff
            })

    return pd.DataFrame.from_records(results)


In [ ]:
subplot_titles = [
    "Admissions", "Medical", "Credit Scoring", "", 
    "Admissions", "Medical", "Judicial (conviction)", "Judicial (penalty)", 
]

fig = make_subplots(rows=2, cols=4, horizontal_spacing=0.05, vertical_spacing=0.2, subplot_titles=subplot_titles, column_widths=[1, 1.2, 1.2, 1.2])

task_list = ["admissions", "diversitymedqa_gender", "south_german", ""]
for i, task_name in enumerate(task_list):
    if task_name == "":
        continue
    model_list = ["Llama-3.1-8B-Instruct", "Ministral-8B-Instruct-2410", "Qwen2.5-7B-Instruct"]
    if task_name == "south_german":
        model_list.append("finance-Llama3-8B")
    elif task_name.startswith("diversitymedqa_"):
        model_list.append("medicine-Llama3-8B")

    blackbox_df = get_blackbox_evaluation_results(concept="gender", task_name=task_name, model_list=model_list)
    whitebox_df = get_steering_evaluation_results(concept="gender", train_dataset="gendered_language", task_name=task_name, model_list=model_list)
    if task_name == "admissions":
        blackbox_df = blackbox_df[blackbox_df.explicit == False]
        
    blackbox_df["model"] = blackbox_df["model"].map(model_label_mapping)
    whitebox_df["model"] = whitebox_df["model"].map(model_label_mapping)
    fig.append_trace(go.Bar(
        x=blackbox_df.model.tolist(), y=blackbox_df.group_diff.tolist(), name="blackbox", marker_color='rgb(55, 83, 109)', showlegend=False
    ), row=1, col=1+i)
    fig.append_trace(go.Bar(
        x=whitebox_df.model.tolist(), y=whitebox_df.slope.tolist(), name="whitebox", marker_color='rgb(26, 118, 255)', showlegend=False
    ), row=1, col=1+i)

task_list = ["admissions", "diversitymedqa_ethnicity", "judicial_guilt", "judicial_penalty"]
for i, task_name in enumerate(task_list):
    if i == 0:
        showlegend = True
    else:
        showlegend = False
    model_list = ["Llama-3.1-8B-Instruct", "Ministral-8B-Instruct-2410", "Qwen2.5-7B-Instruct"]
    if task_name.startswith("judicial_"):
        model_list.append("Saul-7B-Instruct-v1")
    elif task_name.startswith("diversitymedqa_"):
        model_list.append("medicine-Llama3-8B")

    blackbox_df = get_blackbox_evaluation_results(concept="race", task_name=task_name, model_list=model_list)
    whitebox_df = get_steering_evaluation_results(concept="race", train_dataset="dialect", task_name=task_name, model_list=model_list)
    if task_name.startswith("judicial_") or task_name == "admissions":
        blackbox_df = blackbox_df[blackbox_df.explicit == False]

    blackbox_df["model"] = blackbox_df["model"].map(model_label_mapping)
    whitebox_df["model"] = whitebox_df["model"].map(model_label_mapping)
    fig.append_trace(go.Bar(
        x=blackbox_df.model.tolist(), y=blackbox_df.group_diff.tolist(), name="blackbox", marker_color='rgb(55, 83, 109)', showlegend=showlegend
    ), row=2, col=1+i)
    fig.append_trace(go.Bar(
        x=whitebox_df.model.tolist(), y=whitebox_df.slope.tolist(), name="whitebox", marker_color='rgb(26, 118, 255)', showlegend=showlegend
    ), row=2, col=1+i)

fig.update_layout(
    plot_bgcolor='white', width=1100, height=350, barmode='group', bargap=0.25,
    legend=dict(font=dict(size=14), xanchor='left', x=0.82, yanchor='bottom', y=0.75),
    margin=dict(l=25, r=0, t=22, b=15), font=dict(size=13), 
)

fig.update_annotations(font_size=13.8)
fig.update_xaxes(tickfont=dict(size=11), mirror=True, showline=True, linewidth=1, linecolor='darkgrey', tickangle=20)
fig.update_yaxes(tickfont=dict(size=11), mirror=True, showline=True, linewidth=1, linecolor='darkgrey', zeroline = True, zerolinecolor='darkgrey')
fig.update_yaxes(title_text="Gender Bias", title_standoff=8, title_font=dict(size=13), row=1, col=1)
fig.update_yaxes(title_text="Race Bias", title_standoff=8, title_font=dict(size=13), row=2, col=1)

fig.update_xaxes(tickangle=0, col=1)
fig.update_yaxes(range=[-0.02, 0.02], col=1)
fig.update_yaxes(range=[-0.02, 0.02], col=2)
fig.update_yaxes(range=[-0.06, 0.02], row=1, col=3)
fig.update_yaxes(range=[-0.03, 0.2], row=2, col=3, tickvals=[-0.03, 0, 0.05, 0.1, 0.15, 0.2])
fig.update_yaxes(range=[-0.02, 0.04], row=2, col=4)
fig.show()

In [13]:
fig.write_image("plots/overall-blackbox-vs-whitebox.pdf")